# Agentic EDA: ERCOT Load and Weather

This notebook separates deterministic evidence from agent interpretation.
The profiling cells must run without LangGraph, Qwen, or network access.
The optional agent review receives structured findings and proposes follow-up analyses; it does not mutate data.

## Run contract

- Run from the repository root, or set `PROJECT_ROOT` explicitly.
- The preferred input is `data/processed/ercot_hourly_panel/`.
- Use a small fixture while the processed panel is being built.
- Record the input path and artifact version in the saved EDA report.
- Treat agent output as hypotheses until a human approves a follow-up.

In [ ]:
from __future__ import annotations

import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()
INPUT_PATH = PROJECT_ROOT / "data/processed/ercot_hourly_panel"
FIXTURE_PATH = PROJECT_ROOT / "tests/fixtures/eda_panel.parquet"
REPORT_DIR = PROJECT_ROOT / "reports/eda"
RANDOM_SEED = 20260911
sns.set_theme(style="whitegrid")

print({
    "project_root": str(PROJECT_ROOT),
    "input_path": str(INPUT_PATH),
    "input_exists": INPUT_PATH.exists(),
    "fixture_exists": FIXTURE_PATH.exists(),
    "python": platform.python_version(),
    "run_utc": datetime.now(timezone.utc).isoformat(),
})

In [ ]:
def parquet_files(path: Path) -> list[Path]:
    if path.is_file() and path.suffix == ".parquet":
        return [path]
    if path.is_dir():
        return sorted(path.rglob("*.parquet"))
    return []

def load_panel(path: Path, max_files: int = 12) -> tuple[pd.DataFrame, dict]:
    files = parquet_files(path)
    if not files:
        return pd.DataFrame(), {"status": "missing_input", "files": []}
    selected = files[:max_files]
    frame = pd.concat((pd.read_parquet(file) for file in selected), ignore_index=True)
    return frame, {
        "status": "ok",
        "files": [str(file.relative_to(PROJECT_ROOT)) for file in selected],
        "files_available": len(files),
        "files_loaded": len(selected),
    }

panel_path = INPUT_PATH if INPUT_PATH.exists() else FIXTURE_PATH
panel, inventory = load_panel(panel_path)
print(inventory)
if panel.empty:
    print("No panel is available yet. Run the deterministic profiling cells after the processed panel or fixture exists.")
else:
    display(panel.head())

In [ ]:
def profile_panel(frame: pd.DataFrame) -> dict:
    findings = {
        "row_count": int(len(frame)),
        "column_count": int(len(frame.columns)),
        "columns": list(frame.columns),
        "dtypes": {column: str(dtype) for column, dtype in frame.dtypes.items()},
        "missing_fraction": frame.isna().mean().round(6).to_dict(),
    }
    if "timestamp_utc" in frame.columns:
        timestamp = pd.to_datetime(frame["timestamp_utc"], utc=True, errors="coerce")
        findings["timestamp"] = {
            "parse_failures": int(timestamp.isna().sum()),
            "min": None if timestamp.dropna().empty else timestamp.min().isoformat(),
            "max": None if timestamp.dropna().empty else timestamp.max().isoformat(),
            "duplicate_count": int(timestamp.duplicated().sum()),
            "monotonic": bool(timestamp.is_monotonic_increasing),
        }
    for column in ("load_mw", "temperature_regional", "dewpoint_regional", "pressure_regional", "altimeter_regional"):
        if column in frame.columns:
            values = pd.to_numeric(frame[column], errors="coerce").dropna()
            findings.setdefault("numeric_summary", {})[column] = {
                "count": int(values.size),
                "min": None if values.empty else float(values.min()),
                "median": None if values.empty else float(values.median()),
                "max": None if values.empty else float(values.max()),
            }
    return findings

findings = profile_panel(panel)
print(json.dumps(findings, indent=2, default=str))

In [ ]:
if not panel.empty:
    numeric_columns = panel.select_dtypes(include="number").columns
    if len(numeric_columns) > 0:
        display(panel[numeric_columns].describe().T)
        correlation = panel[numeric_columns].corr()
        plt.figure(figsize=(10, 6))
        sns.heatmap(correlation, cmap="vlag", center=0)
        plt.title("Observed numeric correlations")
        plt.tight_layout()
        plt.show()
    if "timestamp_utc" in panel.columns and "load_mw" in panel.columns:
        plot_frame = panel.copy()
        plot_frame["timestamp_utc"] = pd.to_datetime(plot_frame["timestamp_utc"], utc=True, errors="coerce")
        plot_frame = plot_frame.dropna(subset=["timestamp_utc", "load_mw"]).sort_values("timestamp_utc")
        plot_frame = plot_frame.head(24 * 14)
        plt.figure(figsize=(14, 4))
        sns.lineplot(data=plot_frame, x="timestamp_utc", y="load_mw")
        plt.title("Observed load: first loaded two weeks")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## Agent review handoff

The next cell creates the evidence bundle sent to an optional LangGraph/Qwen review graph. The agent should receive facts, not the authority to rewrite the panel. It must label each response as `fact`, `hypothesis`, `risk`, or `approved_followup`.

In [ ]:
evidence_bundle = {
    "input": inventory,
    "findings": findings,
    "analysis_contract": {
        "data_mutation_allowed": False,
        "feature_engineering_allowed": False,
        "causal_claims_allowed": False,
        "human_approval_required_for_followups": True,
    },
}
REPORT_DIR.mkdir(parents=True, exist_ok=True)
evidence_path = REPORT_DIR / "eda_evidence.json"
evidence_path.write_text(json.dumps(evidence_bundle, indent=2, default=str), encoding="utf-8")
print(f"Saved deterministic evidence to {evidence_path}")
print("Optional next step: pass evidence_bundle to the bounded LangGraph review adapter.")

## Human review record

Before accepting an agent suggestion, record:

1. The finding IDs or evidence fields supporting it.
2. Whether it is a fact, hypothesis, risk, or follow-up.
3. The approved analysis, if any.
4. The reason for rejecting or deferring other suggestions.

Persist the reviewed decision under `reports/eda/`; do not rely on transient notebook or chat state.